# 05 — OpenAQ Bronze to Silver

## 1. Read June PM2.5 Measurements

Read OpenAQ JSON data from the Bronze layer and extract individual measurements.

In [ ]:
path = "Files/bronze/openaq/measurements/ccny_pm25_2025-06_page1.json"

In [ ]:
df_openaq = (
    spark.read
    .option("multiline", True)
    .json(path)
)

df_openaq.select("meta.found").show()

In [ ]:
from pyspark.sql.functions import explode
df_measurements = df_openaq.select(
    explode("results").alias("measurement")
)

In [ ]:
print("Total measurements:", df_measurements.count())
df_measurements.printSchema()

## 2. Transform June PM2.5 Measurements

Extract measurement values, pollutant information, sensor ID, and measurement date.

In [ ]:
from pyspark.sql.functions import col

df_air_silver = df_measurements.select(
    col("measurement.value").alias("pm25_value"),
    col("measurement.parameter.name").alias("pollutant"),
    col("measurement.parameter.units").alias("unit"),
    col("measurement.period.datetimeFrom.local").alias("measurement_time_local")
    )

In [ ]:
from pyspark.sql.functions import lit

df_air_silver = df_air_silver.withColumn(
    "sensor_id",
    lit(673)
)

In [ ]:
from pyspark.sql.functions import to_date, substring

df_air_silver = df_air_silver.withColumn(
    "measurement_date",
    to_date(substring("measurement_time_local", 1, 10))
)

In [ ]:
df_air_silver.select(
    "measurement_time_local",
    "measurement_date"
).show(5, truncate=False)

df_air_silver.printSchema()

## 3. Read July PM2.5 Measurements

Read July 2025 OpenAQ measurements from Bronze JSON and extract individual records.

In [ ]:
path_july = "Files/bronze/openaq/measurements/sensor_673_measurements_2025-07_page1.json"

In [ ]:
df_openaq_july = (
    spark.read
    .option("multiline", True)
    .json(path_july)
)

df_openaq_july.select("meta.found").show()

In [ ]:
from pyspark.sql.functions import explode
df_measurements_july = df_openaq_july.select(
    explode("results").alias("measurement")
)

In [ ]:
print("Total July rows:", df_measurements_july.count())
df_measurements_july.printSchema()

## 4. Transform July PM2.5 Measurements

Extract measurement values, pollutant information, sensor ID, and measurement date.

In [ ]:
df_air_july_silver = df_measurements_july.select(
    col("measurement.value").alias("pm25_value"),
    col("measurement.parameter.name").alias("pollutant"),
    col("measurement.parameter.units").alias("unit"),
    col("measurement.period.datetimeFrom.local").alias("measurement_time_local")
)
    


In [ ]:
from pyspark.sql.functions import lit

df_air_july_silver = df_air_july_silver.withColumn(
    "sensor_id",
    lit(673)
)

In [ ]:
from pyspark.sql.functions import to_date, substring

df_air_july_silver = df_air_july_silver.withColumn(
    "measurement_date",
    to_date(substring("measurement_time_local", 1, 10))
)

In [ ]:
df_air_july_silver.select(
    "measurement_time_local",
    "measurement_date"
).show(5, truncate=False)

df_air_july_silver.printSchema()

## 5. Combine June and July PM2.5 Data

Combine monthly PM2.5 measurements into a single Spark DataFrame.

In [ ]:
df_air_combined = df_air_silver.unionByName(df_air_july_silver)

print("Total Combined Rows:", df_air_combined.count())
df_air_combined.groupBy("measurement_date").count().orderBy("measurement_date").show(5)

## 6. Data Quality Checks

Check duplicate measurements, invalid PM2.5 values, and missing measurement dates.

In [ ]:
total_rows = df_air_combined.count()

unique_records = df_air_combined.dropDuplicates(
    ["sensor_id", "measurement_time_local", "pollutant"]
).count()

duplicates = total_rows - unique_records

print("Total Rows: ", total_rows)
print("Unique Records: ", unique_records)
print("Duplicates: ", duplicates)

In [ ]:
invalid_pm25 = df_air_combined.filter(
    (df_air_combined.pm25_value.isNull()) |
    (df_air_combined.pm25_value < 0) 
).count()

print("Invalid pm25: ", invalid_pm25)

In [ ]:
null_dates = df_air_combined.filter(df_air_combined.measurement_date.isNull()).count()

print("NULL Dates: ", null_dates)

## 7. Write Silver Delta Table

Save the combined PM2.5 measurements as a Delta table in the Lakehouse.

In [ ]:
(
    df_air_combined.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_air_quality")
)

## 8. Validate Silver Output

Verify the saved Silver table, row count, and column count.

In [ ]:
df_check = spark.table("silver_air_quality")

print("Rows:", df_check.count())
print("Columns:", len(df_check.columns))